# 01 — Exploratory Data Analysis
## Nepal Rastra Bank (NRB) Macroeconomic Data

**Sections:**
1. Load all datasets
2. Inflation — CPI & WPI trends
3. CPI by Province & Ecology
4. Trade — Imports, Exports, Balance
5. Foreign Exchange — Rate & Reserves
6. Banking — Money Supply, Deposits, Interest Rates
7. Stock Market — NEPSE indicators
8. External Sector — Tourists & Migrants
9. Correlation heatmap of macro features

In [ ]:
import sys, warnings
sys.path.insert(0, '..')   # repo root
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_loader import load_all
from src.preprocessing import (
    build_cpi_series, build_wpi_series,
    build_exchange_rate_series, build_repo_rate_series,
    build_m2_series, build_feature_matrix
)

print('Libraries loaded ✓')

In [ ]:
# ── Load all datasets ──────────────────────────────────────────────────────
data = load_all()

print('Datasets loaded:')
for k, v in data.items():
    shape = v.shape if isinstance(v, pd.DataFrame) else 'N/A'
    status = '✓' if not v.empty else '✗ empty'
    print(f'  {k:25s}  shape={shape}  {status}')

---
## 2. Inflation — CPI & WPI

In [ ]:
cpi = build_cpi_series().dropna()
wpi = build_wpi_series().dropna()

print(f'CPI series: {len(cpi)} months  |  range: {cpi.index.min()} → {cpi.index.max()}')
print(f'WPI series: {len(wpi)} months  |  range: {wpi.index.min()} → {wpi.index.max()}')
print(f'\nCPI stats:\n{cpi.describe().round(2)}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=cpi.index, y=cpi.values, name='CPI YoY',
                          line=dict(color='#c0392b', width=2.5)))
fig.add_trace(go.Scatter(x=wpi.index, y=wpi.values, name='WPI YoY',
                          line=dict(color='#e67e22', width=2, dash='dot')))
fig.add_hrect(y0=5, y1=7, fillcolor='green', opacity=0.07,
               annotation_text='NRB target band 5–7%')
fig.update_layout(title='Nepal CPI vs WPI — Year-over-Year Inflation (%)',
                   yaxis_title='% Change', xaxis_title='Date',
                   template='plotly_white', height=420)
fig.show()

In [ ]:
# CPI by category (heatmap of recent 12 months)
df_cpi = data['cpi_yoy'].copy()
if not df_cpi.empty:
    # Take last 12 periods
    last_12 = df_cpi.iloc[:, -12:]
    fig = px.imshow(
        last_12.fillna(0),
        title='CPI YoY by Category — Last 12 Periods',
        color_continuous_scale='RdYlGn_r',
        aspect='auto',
        labels=dict(color='% Change')
    )
    fig.update_layout(height=500, template='plotly_white')
    fig.show()

---
## 3. CPI by Province & Ecology

In [ ]:
df_prov = data['cpi_province'].copy()
if not df_prov.empty:
    last_6 = df_prov.iloc[:, -6:]
    fig = px.bar(last_6.T, barmode='group',
                  title='Province-wise CPI — Last 6 Periods',
                  labels={'value': 'CPI Index', 'index': 'Period'},
                  template='plotly_white')
    fig.show()

df_eco = data['cpi_ecology'].copy()
if not df_eco.empty:
    last_6 = df_eco.iloc[:, -6:]
    fig2 = px.line(last_6.T, title='Ecology-wise CPI (Mountain / Hill / Terai)',
                    labels={'value': 'CPI Index', 'index': 'Period'},
                    template='plotly_white')
    fig2.show()

---
## 4. Trade Analysis

In [ ]:
df_imp = data['top_imports'].copy()
df_exp = data['top_exports'].copy()

col_imp = 'ten_month_2025_26'
col_exp = 'ten_month_2025_26'

if not df_imp.empty and col_imp in df_imp.columns:
    top_imp = df_imp[col_imp].dropna().sort_values(ascending=False).head(12)
    fig_imp = px.bar(x=top_imp.values, y=top_imp.index,
                      orientation='h',
                      title='Top 12 Import Commodities — 2025/26 (10 months)',
                      labels={'x': 'Rs. million', 'y': 'Commodity'},
                      template='plotly_white', color=top_imp.values,
                      color_continuous_scale='Reds')
    fig_imp.update_yaxes(autorange='reversed')
    fig_imp.show()

if not df_exp.empty and col_exp in df_exp.columns:
    top_exp = df_exp[col_exp].dropna().sort_values(ascending=False).head(12)
    fig_exp = px.bar(x=top_exp.values, y=top_exp.index,
                      orientation='h',
                      title='Top 12 Export Commodities — 2025/26 (10 months)',
                      labels={'x': 'Rs. million', 'y': 'Commodity'},
                      template='plotly_white', color=top_exp.values,
                      color_continuous_scale='Greens')
    fig_exp.update_yaxes(autorange='reversed')
    fig_exp.show()

In [ ]:
# Trade balance over available periods
periods = ['annual_2023_24', 'ten_month_2024_25', 'annual_2024_25', 'ten_month_2025_26']
labels  = ['FY23/24 Annual', 'FY24/25 (10M)', 'FY24/25 Annual', 'FY25/26 (10M)']

imp_totals = [df_imp[p].sum() if p in df_imp.columns else np.nan for p in periods]
exp_totals = [df_exp[p].sum() if p in df_exp.columns else np.nan for p in periods]
balance    = [e - i for e, i in zip(exp_totals, imp_totals)]

fig = go.Figure()
fig.add_bar(x=labels, y=imp_totals, name='Imports', marker_color='#e74c3c')
fig.add_bar(x=labels, y=exp_totals, name='Exports', marker_color='#27ae60')
fig.add_trace(go.Scatter(x=labels, y=balance, name='Trade Balance',
                          mode='lines+markers',
                          line=dict(color='#2c3e50', width=3, dash='dash')))
fig.update_layout(barmode='group', title='Nepal Trade Balance',
                   yaxis_title='Rs. million', template='plotly_white')
fig.show()

---
## 5. Foreign Exchange

In [ ]:
fx = build_exchange_rate_series().dropna()
print(f'Exchange rate series: {len(fx)} months')

fig = px.line(x=fx.index, y=fx.values,
               title='USD/NPR Monthly Exchange Rate',
               labels={'x': 'Date', 'y': 'NPR per USD'},
               template='plotly_white')
fig.update_traces(line_color='#2980b9')
fig.show()

In [ ]:
df_res = data['forex_reserves'].copy()
if not df_res.empty:
    print('Forex Reserves (latest):')
    display(df_res)

---
## 6. Banking — Money Supply & Interest Rates

In [ ]:
m2 = build_m2_series().dropna()
repo = build_repo_rate_series().dropna()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=m2.index, y=m2.values / 1e6, name='M2 (Rs. Trillion)',
                line=dict(color='#8e44ad', width=2)),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=repo.index, y=repo.values, name='Repo Rate (%)',
                line=dict(color='#27ae60', width=2, dash='dot')),
    secondary_y=True
)
fig.update_layout(title='M2 Money Supply vs NRB Repo Rate',
                   template='plotly_white', height=400)
fig.update_yaxes(title_text='M2 (Rs. Trillion)', secondary_y=False)
fig.update_yaxes(title_text='Repo Rate (%)', secondary_y=True)
fig.show()

---
## 7. Stock Market — NEPSE

In [ ]:
df_sm = data['share_market'].copy()
if not df_sm.empty:
    print('NEPSE Stock Market Indicators:')
    display(df_sm)

df_listed = data['listed_companies'].copy()
if not df_listed.empty:
    mktcap_cols = [c for c in df_listed.columns if 'mktcap' in c]
    if mktcap_cols:
        latest = df_listed[mktcap_cols[-1]].dropna()
        fig = px.pie(values=latest.values, names=latest.index,
                      title='Market Cap by Sector (latest)',
                      template='plotly_white')
        fig.show()

---
## 8. External Sector — Tourism & Migrants

In [ ]:
df_t = data['tourist_arrivals'].copy()
if not df_t.empty:
    # Annual totals
    annual = df_t.apply(pd.to_numeric, errors='coerce').sum(axis=0)
    annual.index = [str(i) for i in annual.index]
    annual = annual[annual > 0]
    fig = px.bar(x=annual.index, y=annual.values,
                  title='Annual Tourist Arrivals in Nepal',
                  labels={'x': 'Year', 'y': 'Arrivals'},
                  template='plotly_white', color=annual.values,
                  color_continuous_scale='Blues')
    fig.show()

df_mig = data['migrant_workers'].copy()
if not df_mig.empty:
    latest_col = df_mig.columns[-1]
    top = df_mig[latest_col].dropna().sort_values(ascending=False).head(10)
    fig2 = px.bar(x=top.values, y=top.index, orientation='h',
                   title=f'Top 10 Destination Countries for Migrant Workers ({latest_col})',
                   labels={'x': 'Labour Permits', 'y': 'Country'},
                   template='plotly_white')
    fig2.update_yaxes(autorange='reversed')
    fig2.show()

---
## 9. Feature Matrix & Correlation Heatmap

In [ ]:
df_feat = build_feature_matrix()
print(f'Feature matrix shape: {df_feat.shape}')
print(f'Date range: {df_feat.index.min()} → {df_feat.index.max()}')
df_feat.describe().round(3)

In [ ]:
corr = df_feat.corr().round(2)
fig = px.imshow(corr, text_auto=True,
                 title='Correlation Heatmap — Macroeconomic Features',
                 color_continuous_scale='RdBu_r',
                 zmin=-1, zmax=1,
                 template='plotly_white')
fig.update_layout(height=600)
fig.show()

In [ ]:
# Scatter matrix for the most important features
key_cols = ['cpi_yoy', 'wpi_yoy', 'usd_npr', 'repo_rate', 'cpi_lag_1', 'cpi_lag_12']
available = [c for c in key_cols if c in df_feat.columns]
fig = px.scatter_matrix(df_feat[available].dropna(),
                         dimensions=available,
                         color='cpi_yoy',
                         title='Scatter Matrix — Key Macroeconomic Variables',
                         template='plotly_white',
                         color_continuous_scale='Reds')
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Stationarity test on CPI
from statsmodels.tsa.stattools import adfuller

cpi = build_cpi_series().dropna()
result = adfuller(cpi.values)
print('Augmented Dickey-Fuller Test on CPI YoY')
print(f'  ADF Statistic : {result[0]:.4f}')
print(f'  p-value       : {result[1]:.4f}')
print(f'  Lags used     : {result[2]}')
print(f'  Obs           : {result[3]}')
print('  Critical values:')
for k, v in result[4].items():
    print(f'    {k}: {v:.4f}')

if result[1] < 0.05:
    print('\n✅ Series is stationary (reject H0 at 5%)')
else:
    print('\n⚠️  Series may be non-stationary (fail to reject H0)')